# Build referrence labels to check the output against WAD

## 1. Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd

WAD_ROOT = Path("/Users/amynguyen/Documents/MQ/(26_2) FOSE7902 - Advanced STEM Research Placement/WAD/wad_dataset_v2")
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SUBSET_DIR = REPO_ROOT / "data" / "subsets"
REF_DIR = REPO_ROOT / "data" / "references"
REF_DIR.mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "scripts").mkdir(exist_ok=True)
SEED = 42

## 2. Load the reminders for every clip in the subsets and the test set

In [2]:
ann = pd.concat([pd.read_json(WAD_ROOT / "train_v2.json", lines=True),
                 pd.read_json(WAD_ROOT / "test_alter_v2.json", lines=True)]).drop_duplicates("frame_path")
subsets = pd.concat([pd.read_csv(f)[["frame_path", "subset"]] for f in sorted(SUBSET_DIR.glob("wad_*.csv"))])
df = subsets.merge(ann[["frame_path", "alter"]], on="frame_path", how="left")
df.subset.value_counts().sort_index()

subset
check_01     500
check_02     500
check_03     500
check_04     500
dev_00       500
test        1007
Name: count, dtype: int64

## 3. Parsing rules
The rules are in `src/wad_parser.py`: hazard categories and their words, clock directions, distances in steps, and the action (stop / move / slow / continue).

## 4. Parser every reminder

In [4]:
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from wad_parser import parse

refs = pd.concat([df[["frame_path", "subset"]].reset_index(drop=True),
                  pd.DataFrame([parse(t) for t in df["alter"]])], axis=1)
print("parsed:", round(refs.parsed.mean(), 3), "| path clear:", round(refs.path_clear.mean(), 3),
      "| has distance:", round(refs.primary_dist_steps.notna().mean(), 3))
print("action:", refs.action.value_counts().to_dict())
refs.primary_hazard.value_counts()

parsed: 0.996 | path clear: 0.069 | has distance: 0.11
action: {'continue': 2571, 'slow': 548, 'move': 302, 'stop': 86}


primary_hazard
pedestrian         1098
vehicle             373
pole                269
tree_plant          231
barrier_cone        192
junction            136
wall_door           119
furniture_stall     117
fence_railing       114
sign                112
other_object        105
two_wheeler          94
narrow_crowded       91
street_object        90
off_route            37
stairs_step          32
water                16
curb_edge            16
hole_uneven           6
animal                1
construction          1
Name: count, dtype: int64

## 5. Save the reference labels

In [ ]:
refs.drop(columns="parsed").to_csv(REF_DIR / "wad_references.csv", index=False)
print("saved", REF_DIR / "wad_references.csv", refs.shape)

## 6. Parser accuracy
Rules were developed on dev_00 reminders. After freezing them, 100 random test reminders were checked by hand (1 = correct, 0 = wrong).
The sheet contains WAD text and stays local (gitignored).